# 5. Predict the antigen epitope

For each predicted complex returned by notebook 2, identify the residues on the antigen chain that are in heavy-atom contact (≤ 4.5 Å) with the binder. That set of residues is the *predicted epitope* — the antigen surface the model placed the binder on.

This notebook needs the input CSV (to know each row's antigen sequence) and a folder of predicted CIFs. No ground truth is required.

**Antigen-chain support.** This notebook currently resolves one antigen chain per prediction. Use single-chain antigen inputs for this bundled epitope workflow.

## How it works

For each prediction, we identify the antigen chain and report the antigen residues that contact the binder.

## Outputs

A per-prediction CSV containing the resolved antigen chain ID, the epitope size, and the list of contact residues (both residue numbers as they appear in the CIF and three-letter names). For predicted structures produced from the same single-chain antigen input, these residue IDs can be used directly for external Jaccard / F1 comparison against an experimental epitope.

In [ ]:
import json
import re
import sys
import traceback
from pathlib import Path

import pandas as pd

sys.path.append("src")
from az_adapter import csv_to_jobs
from epitope import predict_epitope, find_antigen_chain_ids

# Path to the input CSV that drove inference in notebook 2.
INPUT_CSV = Path("examples/structure_determination_input.csv")

# Most-recent invoke run under outputs/; override by setting an absolute path.
_invoke_runs = sorted(Path("./outputs").glob("invoke_*"))
PREDICTIONS_DIR = _invoke_runs[-1] / "cifs" if _invoke_runs else Path("./outputs/invoke_<timestamp>/cifs")
OUTPUT_CSV = Path("./outputs/epitope_predictions.csv")

# Heavy-atom contact threshold in Å. 4.5 is a common choice for antibody-antigen contacts.
CONTACT_THRESHOLD_A = 4.5


In [ ]:
# Antigen field per input_name, from the CSV that drove inference. The antigen
# field may list several '/'-separated chains (a multi-chain antigen); keep it
# whole so every antigen chain is resolved and excluded from binder contacts.
import csv as _csv
_csv_rows = list(_csv.DictReader(INPUT_CSV.open(newline="")))
name_to_antigen = {
    job["name"]: row["antigen_seq"]
    for row, job in zip(_csv_rows, csv_to_jobs(INPUT_CSV))
}

# Strip the "batch_NNN_nM_aaK_" prefix that notebook 2 prepends to each CIF.
BATCH_PREFIX_RE = re.compile(r"^batch_\d+_n\d+_aa\d+_")


def input_name_from_cif(cif_path: Path) -> str:
    return BATCH_PREFIX_RE.sub("", cif_path.stem)


prediction_paths = sorted(PREDICTIONS_DIR.glob("*.cif"))
print(f"Found {len(prediction_paths)} prediction CIFs in {PREDICTIONS_DIR}")
if any(input_name_from_cif(p) not in name_to_antigen for p in prediction_paths): raise ValueError("PREDICTIONS_DIR contains CIFs not present in INPUT_CSV; set both paths to the same run.")

rows = []
for cif in prediction_paths:
    input_name = input_name_from_cif(cif)
    antigen_seq = name_to_antigen[input_name]
    try:
        antigen_chain_ids = find_antigen_chain_ids(str(cif), antigen_seq)
        epitope = predict_epitope(
            cif_path=str(cif),
            antigen_chain_ids=antigen_chain_ids,
            contact_threshold_a=CONTACT_THRESHOLD_A,
        )
    except Exception:
        print(f"epitope failed for {cif.name}:")
        traceback.print_exc()
        rows.append({
            "input_name": input_name,
            "prediction_id": cif.stem,
            "antigen_chain_ids": None,
            "epitope_size": None,
            "epitope_chains": None,
            "epitope_residue_ids": None,
            "epitope_residue_names": None,
            "pred_path": str(cif),
        })
        continue
    rows.append({
        "input_name": input_name,
        "prediction_id": cif.stem,
        "antigen_chain_ids": json.dumps(antigen_chain_ids),
        "epitope_size": len(epitope),
        "epitope_chains": json.dumps([r[0] for r in epitope]),
        "epitope_residue_ids": json.dumps([r[1] for r in epitope]),
        "epitope_residue_names": json.dumps([r[2] for r in epitope]),
        "pred_path": str(cif),
    })
    print(f"{input_name}: antigen chains {antigen_chain_ids}, {len(epitope)} epitope residues")

df = pd.DataFrame(rows)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nWrote {len(df)} rows to {OUTPUT_CSV}")
df.head()


## Notes

- **`antigen_chain_id`** is the chain ID that holds the antigen sequence in each predicted CIF.
- **Residue IDs and names** in `epitope_residue_ids` / `epitope_residue_names` are stored as JSON inside the CSV cell so they round-trip cleanly. Read them back with:

  ```python
  pandas.read_csv(
      "epitope_predictions.csv",
      converters={"epitope_residue_ids": json.loads, "epitope_residue_names": json.loads},
  )
  ```
- **The 4.5 Å heavy-atom contact threshold** is a common choice in the antibody-antigen contact-definition literature. Lower it for a tighter epitope, raise it for a broader one — adjust `CONTACT_THRESHOLD_A` above.